In [8]:
import sys
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.saving_utils import DatasetBundle

In [10]:
data_discov_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_discov_full"
)

data_valid_full = DatasetBundle.load(
    project_root / "data/preprocessed/data_valid_full"
)

data_discov_full.meta = cast(
    pd.DataFrame,
    data_discov_full.meta
)

data_valid_full.meta = cast(
    pd.DataFrame,
    data_valid_full.meta
)

Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_discov_full.
Loaded dataset from /Users/aginatore/Desktop/Consulting/esa_consulting/data/preprocessed/data_valid_full.


In [11]:
proba_and_labels_discov = pd.read_parquet(
    project_root
    / "outputs/gmm_clustering/cluster_pred_discov.parquet"
)

proba_and_labels_valid = pd.read_parquet(
    project_root
    / "outputs/gmm_clustering/cluster_pred_valid.parquet"
)

labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display


# ============================================================
# 1. Kontinuierliche und ordinale Variablen auswählen
# ============================================================

profile_variables = (
    data_discov_full.meta
    .loc[
        lambda df: df["type"].isin(["continuous", "ordinal"]),
        "name"
    ]
    .tolist()
)

print(
    f"Laut Metadaten ausgewählt: "
    f"{len(profile_variables)} kontinuierliche/ordinale Variablen"
)


# ============================================================
# 2. Gewichtete Clustermittelwerte aus den Summaries holen
# ============================================================

disc_cluster_means = (
    cluster_discov_summaries
    .filter(regex="_mean$")
    .copy()
)

valid_cluster_means = (
    cluster_valid_summaries
    .filter(regex="_mean$")
    .copy()
)

# "_mean" aus den Spaltennamen entfernen
disc_cluster_means.columns = (
    disc_cluster_means.columns
    .str.replace("_mean", "", regex=False)
)

valid_cluster_means.columns = (
    valid_cluster_means.columns
    .str.replace("_mean", "", regex=False)
)

# Numerisch konvertieren; nicht numerische Einträge werden NaN
disc_cluster_means = disc_cluster_means.apply(
    pd.to_numeric,
    errors="coerce"
)

valid_cluster_means = valid_cluster_means.apply(
    pd.to_numeric,
    errors="coerce"
)

# Gewichtungsvariable sicherheitshalber entfernen
disc_cluster_means = disc_cluster_means.drop(
    columns=["wichtn"],
    errors="ignore"
)

valid_cluster_means = valid_cluster_means.drop(
    columns=["wichtn"],
    errors="ignore"
)

# Nur ausgewählte Variablen verwenden, die überall vorhanden sind
common_variables = [
    variable
    for variable in profile_variables
    if variable in disc_cluster_means.columns
    and variable in valid_cluster_means.columns
    and variable in data_discov_cont.columns
]

disc_cluster_means = disc_cluster_means[common_variables]
valid_cluster_means = valid_cluster_means[common_variables]

print(
    f"In beiden Cluster-Summaries vorhanden: "
    f"{len(common_variables)} Variablen"
)


# ============================================================
# 3. Discovery-Personendaten vorbereiten
# ============================================================

disc_person_data = (
    data_discov_cont[common_variables]
    .apply(pd.to_numeric, errors="coerce")
)

disc_weights = np.asarray(
    sample_weight_discov,
    dtype=float
)

if len(disc_person_data) != len(disc_weights):
    raise ValueError(
        "Discovery-Personendaten und Gewichte haben "
        "unterschiedliche Längen."
    )


# ============================================================
# 4. Gewichteten Discovery-Mittelwert und gewichtete SD
#    für jede Variable berechnen
# ============================================================

def weighted_mean_sd_safe(data, weights):
    """
    Berechnet pro Variable den gewichteten Mittelwert und
    die gewichtete Populationsstandardabweichung.

    Fehlende Werte und nicht positive/ungültige Gewichte
    werden ausgeschlossen.
    """
    weights = np.asarray(weights, dtype=float)

    means = {}
    standard_deviations = {}
    excluded = {}

    for variable in data.columns:
        values = pd.to_numeric(
            data[variable],
            errors="coerce"
        ).to_numpy(dtype=float)

        valid = (
            np.isfinite(values)
            & np.isfinite(weights)
            & (weights > 0)
        )

        x = values[valid]
        w = weights[valid]

        if len(x) == 0 or w.sum() <= 0:
            means[variable] = np.nan
            standard_deviations[variable] = np.nan
            excluded[variable] = (
                "keine gültigen Werte mit positivem Gewicht"
            )
            continue

        mean = np.average(x, weights=w)

        variance = np.average(
            (x - mean) ** 2,
            weights=w
        )

        sd = np.sqrt(variance)

        means[variable] = mean
        standard_deviations[variable] = sd

        if not np.isfinite(sd):
            excluded[variable] = "SD nicht definiert"
        elif sd == 0:
            excluded[variable] = "keine Varianz"

    return (
        pd.Series(means, dtype=float),
        pd.Series(standard_deviations, dtype=float),
        excluded
    )


discovery_mean, discovery_sd, excluded_variables = (
    weighted_mean_sd_safe(
        disc_person_data,
        disc_weights
    )
)

print("\nVariablen ohne verwendbare Discovery-SD:")

if excluded_variables:
    display(
        pd.Series(
            excluded_variables,
            name="Grund"
        ).to_frame()
    )
else:
    print("Keine")


# ============================================================
# 5. Nur Variablen mit gültiger positiver Discovery-SD behalten
# ============================================================

usable_variables = [
    variable
    for variable in common_variables
    if pd.notna(discovery_mean[variable])
    and pd.notna(discovery_sd[variable])
    and np.isfinite(discovery_sd[variable])
    and discovery_sd[variable] > 0
]

print(f"\nVerwendete Variablen: {len(usable_variables)}")
print(
    f"Entfernte Variablen: "
    f"{len(common_variables) - len(usable_variables)}"
)

if len(usable_variables) < 3:
    raise ValueError(
        "Für den Profilvergleich sind weniger als drei "
        "verwendbare Variablen vorhanden."
    )


# ============================================================
# 6. Discovery- und Validation-Clusterprofile mit derselben
#    Discovery-Skala standardisieren
# ============================================================

disc_profiles_z = (
    disc_cluster_means[usable_variables]
    - discovery_mean[usable_variables]
) / discovery_sd[usable_variables]

valid_profiles_z = (
    valid_cluster_means[usable_variables]
    - discovery_mean[usable_variables]
) / discovery_sd[usable_variables]

print("\nStandardisierte Discovery-Clusterprofile:")
display(disc_profiles_z.round(3))

print("Standardisierte Validation-Clusterprofile:")
display(valid_profiles_z.round(3))


# ============================================================
# 7. Spearman-Ähnlichkeitsmatrix berechnen
# ============================================================

spearman_matrix = pd.DataFrame(
    index=[
        f"Discovery {cluster}"
        for cluster in disc_profiles_z.index
    ],
    columns=[
        f"Validation {cluster}"
        for cluster in valid_profiles_z.index
    ],
    dtype=float
)

p_value_matrix = spearman_matrix.copy()
n_variables_matrix = spearman_matrix.copy()

for discovery_cluster in disc_profiles_z.index:
    for validation_cluster in valid_profiles_z.index:

        discovery_profile = disc_profiles_z.loc[
            discovery_cluster
        ]

        validation_profile = valid_profiles_z.loc[
            validation_cluster
        ]

        # Nur Variablen verwenden, die in beiden Profilen gültig sind
        complete = (
            discovery_profile.notna()
            & validation_profile.notna()
        )

        x = discovery_profile.loc[complete]
        y = validation_profile.loc[complete]

        row_name = f"Discovery {discovery_cluster}"
        column_name = f"Validation {validation_cluster}"

        n_variables_matrix.loc[
            row_name,
            column_name
        ] = len(x)

        if (
            len(x) < 3
            or x.nunique() < 2
            or y.nunique() < 2
        ):
            rho = np.nan
            p_value = np.nan
        else:
            rho, p_value = spearmanr(x, y)

        spearman_matrix.loc[
            row_name,
            column_name
        ] = rho

        p_value_matrix.loc[
            row_name,
            column_name
        ] = p_value


print("\nSpearman-Ähnlichkeitsmatrix der Clusterprofile:")
display(spearman_matrix.round(3))

print("Anzahl der jeweils verwendeten Variablen:")
display(n_variables_matrix.astype("Int64"))
